# dojo

This notebook scans all `markov_blanket_detection_summary.csv` files under `RESULTS/synthetic` and analyzes when the `_MS_linear` method reaches the best `mean_recall_mb_grouped`.

It contains:
- a general analysis for `mRMR_MS_linear` against all methods present in each CSV,
- a focused analysis for `MS` vs `mRMR` vs `mRMR_MS_linear`,
- a focused analysis for `MS` vs `JMI` vs `JMI_MS_linear`,
- dataset-level structural tables that relate performance to network characteristics.


In [ ]:
from pathlib import Path
import re

import pandas as pd
from IPython.display import display

pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 200)

RESULTS_ROOT = Path('/mnt/storage/mlopezdecas/MRMR/new_experiments/RESULTS/synthetic')
RUN_PATTERN = re.compile(r'^(?P<network>.+?)__target_(?P<target>.+?)__n_(?P<n_samples>\d+)__rep_(?P<rep>\d+)$')


def load_summary_records(results_root: Path = RESULTS_ROOT) -> pd.DataFrame:
    rows = []
    for csv_path in sorted(results_root.glob('save_*/*/results/markov_blanket_detection_summary.csv')):
        run_folder = csv_path.parent.parent.name
        run_match = RUN_PATTERN.match(run_folder)
        if run_match is None:
            continue

        df = pd.read_csv(csv_path)
        if 'method_output_name' not in df.columns or 'mean_recall_mb_grouped' not in df.columns:
            continue

        df = df[['method_output_name', 'mean_recall_mb_grouped']].copy()
        df['mean_recall_mb_grouped'] = pd.to_numeric(df['mean_recall_mb_grouped'], errors='coerce')
        df = df.dropna(subset=['method_output_name', 'mean_recall_mb_grouped'])

        for _, row in df.iterrows():
            rows.append(
                {
                    'dataset_name': csv_path.parents[2].name.replace('save_', ''),
                    'run_folder': run_folder,
                    'network_name': run_match.group('network'),
                    'target_name': run_match.group('target'),
                    'n_samples': int(run_match.group('n_samples')),
                    'replication_id': int(run_match.group('rep')),
                    'method_output_name': row['method_output_name'],
                    'mean_recall_mb_grouped': float(row['mean_recall_mb_grouped']),
                    'path': str(csv_path),
                }
            )

    return pd.DataFrame(rows)


def analyze_method_best(summary_df: pd.DataFrame, target_method: str, allowed_methods: list[str] | None = None):
    if allowed_methods is not None:
        working_df = summary_df.loc[summary_df['method_output_name'].isin(allowed_methods)].copy()
    else:
        working_df = summary_df.copy()

    grouped = []
    group_cols = ['dataset_name', 'run_folder', 'network_name', 'target_name', 'n_samples', 'replication_id', 'path']
    for group_key, group_df in working_df.groupby(group_cols, sort=True):
        if target_method not in set(group_df['method_output_name']):
            continue

        target_score = float(group_df.loc[group_df['method_output_name'] == target_method, 'mean_recall_mb_grouped'].iloc[0])
        best_score = float(group_df['mean_recall_mb_grouped'].max())
        best_methods = sorted(group_df.loc[group_df['mean_recall_mb_grouped'] == best_score, 'method_output_name'].tolist())

        if target_score == best_score and best_methods == [target_method]:
            status = 'strict_best'
        elif target_score == best_score:
            status = 'tied_best'
        else:
            status = 'not_best'

        grouped.append(
            {
                'dataset_name': group_key[0],
                'run_folder': group_key[1],
                'network_name': group_key[2],
                'target_name': group_key[3],
                'n_samples': group_key[4],
                'replication_id': group_key[5],
                'path': group_key[6],
                'target_method': target_method,
                'target_score': target_score,
                'best_score': best_score,
                'best_methods': ', '.join(best_methods),
                'status': status,
                'methods_considered': ', '.join(sorted(group_df['method_output_name'].tolist())),
            }
        )

    results_df = pd.DataFrame(grouped).sort_values(['status', 'dataset_name', 'n_samples', 'run_folder']).reset_index(drop=True)
    summary_by_samples = (
        results_df.groupby(['status', 'n_samples'])
        .size()
        .rename('n_csvs')
        .reset_index()
        .sort_values(['status', 'n_samples'])
    ) if not results_df.empty else pd.DataFrame(columns=['status', 'n_samples', 'n_csvs'])

    strict_best_df = results_df.loc[results_df['status'] == 'strict_best'].copy()
    tied_best_df = results_df.loc[results_df['status'] == 'tied_best'].copy()
    not_best_df = results_df.loc[results_df['status'] == 'not_best'].copy()

    return {
        'results_df': results_df,
        'summary_by_samples': summary_by_samples,
        'strict_best_df': strict_best_df,
        'tied_best_df': tied_best_df,
        'not_best_df': not_best_df,
    }


summary_df = load_summary_records()
EXCLUDED_DATASETS = {'diabetes'}
summary_df = summary_df.loc[~summary_df['dataset_name'].isin(EXCLUDED_DATASETS)].copy()
print(f'Excluded datasets: {sorted(EXCLUDED_DATASETS)}')
print(f'Total method rows loaded: {len(summary_df)}')
print(f'Total CSV files loaded: {summary_df['path'].nunique()}')


## General analysis for `mRMR_MS_linear`

This reproduces the original notebook behavior: `mRMR_MS_linear` is checked against all methods available in each CSV.


In [ ]:
general_mrmr_ms = analyze_method_best(summary_df, target_method='mRMR_MS_linear')

print(f"Total CSVs analyzed: {len(general_mrmr_ms['results_df'])}")
print(f"MRMR-MS strict best: {len(general_mrmr_ms['strict_best_df'])}")
print(f"MRMR-MS tied best:   {len(general_mrmr_ms['tied_best_df'])}")
print(f"MRMR-MS not best:    {len(general_mrmr_ms['not_best_df'])}")

display(general_mrmr_ms['summary_by_samples'])


In [ ]:
display(
    general_mrmr_ms['strict_best_df'][
        [
            'dataset_name',
            'network_name',
            'target_name',
            'n_samples',
            'replication_id',
            'target_score',
            'best_score',
            'path',
        ]
    ]
)


In [ ]:
display(
    general_mrmr_ms['tied_best_df'][
        [
            'dataset_name',
            'network_name',
            'target_name',
            'n_samples',
            'replication_id',
            'target_score',
            'best_methods',
            'path',
        ]
    ]
)


## Focused analysis: `MS` vs `mRMR` vs `mRMR-MS`

Here the comparison is restricted to the triplet `MS`, `mRMR`, and `mRMR_MS_linear` only.


In [ ]:
mrmr_triplet = analyze_method_best(
    summary_df,
    target_method='mRMR_MS_linear',
    allowed_methods=['MS', 'mRMR', 'mRMR_MS_linear'],
)

print(f"Total CSVs analyzed: {len(mrmr_triplet['results_df'])}")
print(f"MRMR-MS strict best: {len(mrmr_triplet['strict_best_df'])}")
print(f"MRMR-MS tied best:   {len(mrmr_triplet['tied_best_df'])}")
print(f"MRMR-MS not best:    {len(mrmr_triplet['not_best_df'])}")

display(mrmr_triplet['summary_by_samples'])


In [ ]:
display(
    mrmr_triplet['strict_best_df'][
        [
            'dataset_name',
            'network_name',
            'target_name',
            'n_samples',
            'replication_id',
            'target_score',
            'best_score',
            'path',
        ]
    ]
)


In [ ]:
display(
    mrmr_triplet['tied_best_df'][
        [
            'dataset_name',
            'network_name',
            'target_name',
            'n_samples',
            'replication_id',
            'target_score',
            'best_methods',
            'path',
        ]
    ]
)


## Focused analysis: `MS` vs `JMI` vs `JMI-MS`

Here the comparison is restricted to the triplet `MS`, `JMI`, and `JMI_MS_linear` only.


In [ ]:
jmi_triplet = analyze_method_best(
    summary_df,
    target_method='JMI_MS_linear',
    allowed_methods=['MS', 'JMI', 'JMI_MS_linear'],
)

print(f"Total CSVs analyzed: {len(jmi_triplet['results_df'])}")
print(f"JMI-MS strict best: {len(jmi_triplet['strict_best_df'])}")
print(f"JMI-MS tied best:   {len(jmi_triplet['tied_best_df'])}")
print(f"JMI-MS not best:    {len(jmi_triplet['not_best_df'])}")

display(jmi_triplet['summary_by_samples'])


In [ ]:
display(
    jmi_triplet['strict_best_df'][
        [
            'dataset_name',
            'network_name',
            'target_name',
            'n_samples',
            'replication_id',
            'target_score',
            'best_score',
            'path',
        ]
    ]
)


In [ ]:
display(
    jmi_triplet['tied_best_df'][
        [
            'dataset_name',
            'network_name',
            'target_name',
            'n_samples',
            'replication_id',
            'target_score',
            'best_methods',
            'path',
        ]
    ]
)


## Dataset-level tables for paper discussion

The following tables recompute the current contents of `RESULTS/synthetic` and relate the behavior of the `_MS_linear` variants to structural properties of each Bayesian network and to the dimensionality induced by one-hot encoding.

`best_or_tie_rate` denotes the fraction of analyzed `(dataset, target, sample size)` configurations where the target method is either strictly best or tied for the best `mean_recall_mb_grouped`.


In [ ]:
import csv
import json

BNLEARN_METADATA_ROOT = Path('/mnt/storage/mlopezdecas/MRMR/new_experiments/BNlearn/synthetic_datasets_rds')
DATA_ROOT = Path('/mnt/storage/mlopezdecas/MRMR/new_experiments/DATA/synthetic')


def load_dataset_characteristics(summary_df: pd.DataFrame) -> pd.DataFrame:
    dataset_rows = []
    datasets = sorted(summary_df['dataset_name'].dropna().unique())

    for dataset_name in datasets:
        metadata_path = BNLEARN_METADATA_ROOT / dataset_name / 'network_metadata.json'
        targets_path = BNLEARN_METADATA_ROOT / dataset_name / 'summary_targets.csv'
        metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
        targets_df = pd.read_csv(targets_path)

        targets_in_results = sorted(
            summary_df.loc[summary_df['dataset_name'] == dataset_name, 'target_name']
            .astype(str)
            .unique()
            .tolist()
        )
        targets_used_df = targets_df.loc[targets_df['target'].astype(str).isin(targets_in_results)].copy()
        if targets_used_df.empty:
            targets_used_df = targets_df.copy()

        feature_counts = []
        onehot_var_counts = []
        onehot_dummy_counts = []
        max_dummy_widths = []
        for csv_path in sorted((DATA_ROOT / dataset_name).glob('*__n_250__rep_1.csv')):
            with csv_path.open('r', encoding='utf-8', newline='') as f:
                reader = csv.reader(f)
                header = next(reader)
            feature_counts.append(max(len(header) - 1, 0))

            onehot_path = csv_path.with_name(f'{csv_path.stem}__onehot_metadata.json')
            if onehot_path.exists():
                payload = json.loads(onehot_path.read_text(encoding='utf-8'))
                mappings = payload.get('one_hot_mappings', {})
                widths = [len(cols) for cols in mappings.values() if isinstance(cols, list)]
                onehot_var_counts.append(len(widths))
                onehot_dummy_counts.append(sum(widths))
                max_dummy_widths.append(max(widths) if widths else 0)
            else:
                onehot_var_counts.append(0)
                onehot_dummy_counts.append(0)
                max_dummy_widths.append(0)

        dataset_rows.append(
            {
                'dataset_name': dataset_name,
                'n_nodes': int(metadata['n_nodes']),
                'n_targets_eval': len(targets_in_results),
                'mb_mean': float(pd.to_numeric(targets_used_df['mb_size']).mean()),
                'mb_median': float(pd.to_numeric(targets_used_df['mb_size']).median()),
                'parents_mean': float(pd.to_numeric(targets_used_df['n_parents']).mean()),
                'children_mean': float(pd.to_numeric(targets_used_df['n_children']).mean()),
                'spouses_mean': float(pd.to_numeric(targets_used_df['n_spouses']).mean()),
                'features_after_ohe_mean': float(pd.Series(feature_counts, dtype=float).mean()),
                'onehot_source_vars_mean': float(pd.Series(onehot_var_counts, dtype=float).mean()),
                'onehot_dummy_cols_mean': float(pd.Series(onehot_dummy_counts, dtype=float).mean()),
                'max_dummy_width_mean': float(pd.Series(max_dummy_widths, dtype=float).mean()),
            }
        )

    out = pd.DataFrame(dataset_rows).sort_values('dataset_name').reset_index(drop=True)
    numeric_cols = [
        'mb_mean',
        'mb_median',
        'parents_mean',
        'children_mean',
        'spouses_mean',
        'features_after_ohe_mean',
        'onehot_source_vars_mean',
        'onehot_dummy_cols_mean',
        'max_dummy_width_mean',
    ]
    out[numeric_cols] = out[numeric_cols].round(3)
    return out


def build_dataset_performance_table(results_df: pd.DataFrame, dataset_characteristics_df: pd.DataFrame) -> pd.DataFrame:
    working = results_df.copy()
    working['best_or_tie'] = working['status'] != 'not_best'

    status_counts = (
        working.groupby(['dataset_name', 'status'])
        .size()
        .unstack(fill_value=0)
        .reset_index()
    )
    for col in ['strict_best', 'tied_best', 'not_best']:
        if col not in status_counts.columns:
            status_counts[col] = 0

    best_or_tie_rate = (
        working.groupby('dataset_name')['best_or_tie']
        .mean()
        .rename('best_or_tie_rate')
        .reset_index()
    )

    best_or_tie_by_samples = (
        working.groupby(['dataset_name', 'n_samples'])['best_or_tie']
        .mean()
        .rename('best_or_tie_rate')
        .reset_index()
        .pivot(index='dataset_name', columns='n_samples', values='best_or_tie_rate')
        .rename(columns={250: 'best_or_tie_250', 1000: 'best_or_tie_1000', 5000: 'best_or_tie_5000'})
        .reset_index()
    )

    out = dataset_characteristics_df.merge(best_or_tie_rate, on='dataset_name', how='left')
    out = out.merge(
        status_counts[['dataset_name', 'strict_best', 'tied_best', 'not_best']],
        on='dataset_name',
        how='left',
    )
    out = out.merge(best_or_tie_by_samples, on='dataset_name', how='left')

    for col in ['best_or_tie_rate', 'best_or_tie_250', 'best_or_tie_1000', 'best_or_tie_5000']:
        if col in out.columns:
            out[col] = out[col].fillna(0.0).round(3)
    for col in ['strict_best', 'tied_best', 'not_best']:
        out[col] = out[col].fillna(0).astype(int)

    return out.sort_values('best_or_tie_rate', ascending=False).reset_index(drop=True)


def build_extremes_table(profile_df: pd.DataFrame, method_label: str, top_n: int = 5) -> pd.DataFrame:
    columns = [
        'dataset_name',
        'best_or_tie_rate',
        'best_or_tie_250',
        'best_or_tie_1000',
        'best_or_tie_5000',
        'n_nodes',
        'mb_mean',
        'spouses_mean',
        'features_after_ohe_mean',
        'onehot_dummy_cols_mean',
    ]
    top = profile_df.loc[:, columns].head(top_n).copy()
    top.insert(0, 'group', 'top')
    top.insert(1, 'target_method', method_label)

    bottom = profile_df.loc[:, columns].tail(top_n).iloc[::-1].copy()
    bottom.insert(0, 'group', 'bottom')
    bottom.insert(1, 'target_method', method_label)

    return pd.concat([top, bottom], ignore_index=True)


dataset_characteristics_df = load_dataset_characteristics(summary_df)
dataset_characteristics_table = dataset_characteristics_df[
    [
        'dataset_name',
        'n_nodes',
        'n_targets_eval',
        'mb_mean',
        'mb_median',
        'parents_mean',
        'children_mean',
        'spouses_mean',
        'features_after_ohe_mean',
        'onehot_source_vars_mean',
        'onehot_dummy_cols_mean',
        'max_dummy_width_mean',
    ]
].sort_values('dataset_name').reset_index(drop=True)

mrmr_dataset_profile_df = build_dataset_performance_table(mrmr_triplet['results_df'], dataset_characteristics_df)
mrmr_dataset_profile_table = mrmr_dataset_profile_df[
    [
        'dataset_name',
        'best_or_tie_rate',
        'strict_best',
        'tied_best',
        'not_best',
        'best_or_tie_250',
        'best_or_tie_1000',
        'best_or_tie_5000',
        'n_nodes',
        'mb_mean',
        'spouses_mean',
        'features_after_ohe_mean',
        'onehot_dummy_cols_mean',
    ]
]

jmi_dataset_profile_df = build_dataset_performance_table(jmi_triplet['results_df'], dataset_characteristics_df)
jmi_dataset_profile_table = jmi_dataset_profile_df[
    [
        'dataset_name',
        'best_or_tie_rate',
        'strict_best',
        'tied_best',
        'not_best',
        'best_or_tie_250',
        'best_or_tie_1000',
        'best_or_tie_5000',
        'n_nodes',
        'mb_mean',
        'spouses_mean',
        'features_after_ohe_mean',
        'onehot_dummy_cols_mean',
    ]
]

mrmr_extremes_df = build_extremes_table(mrmr_dataset_profile_table, 'mRMR_MS_linear')
jmi_extremes_df = build_extremes_table(jmi_dataset_profile_table, 'JMI_MS_linear')


In [ ]:
display(dataset_characteristics_table)

In [ ]:
display(mrmr_dataset_profile_table)
display(mrmr_extremes_df)

In [ ]:
display(jmi_dataset_profile_table)
display(jmi_extremes_df)

### LaTeX exports

These strings can be copied directly into the manuscript or written to `.tex` files from the optional export cell.


In [ ]:
latex_float = lambda x: f'{x:.3f}'

latex_exports = {
    'dataset_characteristics_table': dataset_characteristics_table.to_latex(index=False, float_format=latex_float),
    'mrmr_dataset_profile_table': mrmr_dataset_profile_table.to_latex(index=False, float_format=latex_float),
    'jmi_dataset_profile_table': jmi_dataset_profile_table.to_latex(index=False, float_format=latex_float),
    'mrmr_extremes_df': mrmr_extremes_df.to_latex(index=False, float_format=latex_float),
    'jmi_extremes_df': jmi_extremes_df.to_latex(index=False, float_format=latex_float),
}

for name, latex_table in latex_exports.items():
    print(f'===== {name} =====')
    print(latex_table)


===== dataset_characteristics_table =====
\begin{tabular}{lrrrrrrrrrrr}
\toprule
dataset_name & n_nodes & n_targets_eval & mb_mean & mb_median & parents_mean & children_mean & spouses_mean & features_after_ohe_mean & onehot_source_vars_mean & onehot_dummy_cols_mean & max_dummy_width_mean \\
\midrule
arth150 & 107 & 10 & 7.100 & 7.000 & 3.100 & 2.500 & 4.300 & 106.000 & 0.000 & 0.000 & 0.000 \\
ecoli70 & 46 & 10 & 6.700 & 6.500 & 1.800 & 3.400 & 3.100 & 45.000 & 0.000 & 0.000 & 0.000 \\
healthcare & 7 & 4 & 3.250 & 3.500 & 1.500 & 1.000 & 1.000 & 8.000 & 2.000 & 4.000 & 2.000 \\
link & 724 & 10 & 12.000 & 9.000 & 3.000 & 4.200 & 4.800 & 609.200 & 0.600 & 1.200 & 0.400 \\
magic-irri & 64 & 10 & 11.500 & 7.500 & 2.300 & 2.000 & 8.400 & 63.000 & 0.000 & 0.000 & 0.000 \\
magic-niab & 44 & 10 & 12.500 & 11.000 & 2.000 & 1.900 & 9.300 & 43.000 & 0.000 & 0.000 & 0.000 \\
mehra-complete & 24 & 10 & 16.800 & 16.500 & 5.800 & 2.400 & 13.700 & 63.200 & 4.000 & 44.200 & 28.200 \\
pathfinder & 109 &

### Interpretation notes

- `mRMR_MS_linear` performs best on `mehra-complete`, `pathfinder`, `link`, and `ecoli70` after excluding `diabetes`. These favorable cases correspond either to post-encoding dimensionality expansion (`mehra-complete`, `pathfinder`) or to medium-sized networks with comparatively clean local structure (`ecoli70`).
- The weakest regimes for `mRMR_MS_linear` are `magic-niab`, `healthcare`, and `magic-irri`. Two unfavorable profiles emerge: very small networks with limited room for separation (`healthcare`) and spouse-heavy networks without substantial one-hot expansion (`magic-niab`, `magic-irri`).
- `sangiovese` is locally dense (`mb_mean` and `spouses_mean` are both high) yet remains difficult for both hybrid variants, suggesting that very dense local neighborhoods reduce the benefit of the strangeness-based refinement.
- `JMI_MS_linear` shows a different behavior on `link`: `mRMR_MS_linear` remains competitive, but `JMI_MS_linear` almost never wins. This indicates that the gain depends not only on dataset size, but also on the interaction between the scoring family and the network structure.


## Optional exports


In [ ]:
# mrmr_triplet['strict_best_df'].to_csv('dojo_mrmr_triplet_strict_best.csv', index=False)
# mrmr_triplet['tied_best_df'].to_csv('dojo_mrmr_triplet_tied_best.csv', index=False)
# jmi_triplet['strict_best_df'].to_csv('dojo_jmi_triplet_strict_best.csv', index=False)
# jmi_triplet['tied_best_df'].to_csv('dojo_jmi_triplet_tied_best.csv', index=False)
# dataset_characteristics_table.to_csv('dojo_dataset_characteristics_table.csv', index=False)
# mrmr_dataset_profile_table.to_csv('dojo_mrmr_dataset_profile_table.csv', index=False)
# jmi_dataset_profile_table.to_csv('dojo_jmi_dataset_profile_table.csv', index=False)
# mrmr_extremes_df.to_csv('dojo_mrmr_dataset_extremes.csv', index=False)
# jmi_extremes_df.to_csv('dojo_jmi_dataset_extremes.csv', index=False)
# Path('dojo_dataset_characteristics_table.tex').write_text(dataset_characteristics_table.to_latex(index=False, float_format=lambda x: f'{x:.3f}'), encoding='utf-8')
# Path('dojo_mrmr_dataset_profile_table.tex').write_text(mrmr_dataset_profile_table.to_latex(index=False, float_format=lambda x: f'{x:.3f}'), encoding='utf-8')
# Path('dojo_jmi_dataset_profile_table.tex').write_text(jmi_dataset_profile_table.to_latex(index=False, float_format=lambda x: f'{x:.3f}'), encoding='utf-8')
# Path('dojo_mrmr_dataset_extremes.tex').write_text(mrmr_extremes_df.to_latex(index=False, float_format=lambda x: f'{x:.3f}'), encoding='utf-8')
# Path('dojo_jmi_dataset_extremes.tex').write_text(jmi_extremes_df.to_latex(index=False, float_format=lambda x: f'{x:.3f}'), encoding='utf-8')


In [ ]:
# Supplementary LaTeX tables for the mRMR-MS dataset-level discussion
# Focus: mRMR_MS_linear compared only with its components, MS and mRMR.

DATASET_DISPLAY_ORDER = [
    'link',
    'pathfinder',
    'ecoli70',
    'magic-niab',
    'magic-irri',
    'arth150',
    'healthcare',
    'sangiovese',
    'mehra-complete',
]
DATASET_DISPLAY_NAMES = {
    'link': 'Link',
    'pathfinder': 'Pathfinder',
    'ecoli70': 'Ecoli70',
    'magic-niab': 'Magic-NIAB',
    'magic-irri': 'Magic-IRRI',
    'arth150': 'Arth150',
    'healthcare': 'Healthcare',
    'sangiovese': 'Sangiovese',
    'mehra-complete': 'Mehra',
}
MRMR_DISCUSSION_DATASETS = DATASET_DISPLAY_ORDER.copy()


def _latex_float_3(value):
    return f'{value:.3f}'


def _dataset_order_rank(values):
    order = {name: rank for rank, name in enumerate(DATASET_DISPLAY_ORDER)}
    return values.map(order).fillna(len(order)).astype(int)


def _sort_by_dataset_order(df, dataset_col='dataset_name'):
    out = df.copy()
    out['_dataset_order'] = _dataset_order_rank(out[dataset_col])
    out = out.sort_values('_dataset_order').drop(columns='_dataset_order').reset_index(drop=True)
    return out


def _display_dataset_names(df, dataset_col='dataset_name', output_col='Dataset'):
    out = df.copy()
    out[dataset_col] = out[dataset_col].map(DATASET_DISPLAY_NAMES).fillna(out[dataset_col])
    return out.rename(columns={dataset_col: output_col})


def _clean_metric_columns(df):
    return df.rename(
        columns={
            'best_or_tie_rate': 'Best/tie',
            'strict_best': 'Strict best',
            'tied_best': 'Tied best',
            'not_best': 'Not best',
            'best_or_tie_250': 'n=250',
            'best_or_tie_1000': 'n=1000',
            'best_or_tie_5000': 'n=5000',
            'n_nodes': 'Nodes',
            'mb_mean': 'MB mean',
            'spouses_mean': 'Spouses mean',
            'features_after_ohe_mean': 'OHE features',
            'onehot_dummy_cols_mean': 'OHE dummies',
            'discussion_group': 'Profile',
            'win_or_tie_vs_MS': 'Win/tie vs MS',
            'win_or_tie_vs_mRMR': 'Win/tie vs mRMR',
        }
    )


def _publishable_table(df, dataset_col='dataset_name'):
    return _clean_metric_columns(_display_dataset_names(_sort_by_dataset_order(df, dataset_col), dataset_col))


def _print_latex_table(name, df, caption, label, column_format=None, font_size='small'):
    print(f'===== {name} =====')
    tabular = df.to_latex(
        index=False,
        float_format=_latex_float_3,
        escape=False,
        multicolumn=True,
        multicolumn_format='c',
        column_format=column_format,
    )
    print('\\begin{table}[!htbp]')
    print('\\centering')
    print(f'\\{font_size}')
    print('\\setlength{\\tabcolsep}{4pt}')
    print('\\renewcommand{\\arraystretch}{1.08}')
    print(f'\\caption{{{caption}}}')
    print(f'\\label{{{label}}}')
    print(tabular, end='')
    print('\\end{table}')
    print()


def build_component_pairwise_summary(summary_df, target_method='mRMR_MS_linear', components=('MS', 'mRMR')):
    allowed_methods = [*components, target_method]
    working = summary_df.loc[summary_df['method_output_name'].isin(allowed_methods)].copy()
    unit_cols = [
        'dataset_name',
        'run_folder',
        'network_name',
        'target_name',
        'n_samples',
        'replication_id',
    ]
    pivot = working.pivot_table(
        index=unit_cols,
        columns='method_output_name',
        values='mean_recall_mb_grouped',
        aggfunc='first',
    ).reset_index()

    rows = []
    for component in components:
        paired = pivot.dropna(subset=[target_method, component]).copy()
        paired['delta'] = paired[target_method] - paired[component]
        for dataset_name, chunk in paired.groupby('dataset_name', sort=False):
            wins = int((chunk['delta'] > 0).sum())
            ties = int((chunk['delta'] == 0).sum())
            losses = int((chunk['delta'] < 0).sum())
            total = wins + ties + losses
            rows.append(
                {
                    'dataset_name': dataset_name,
                    'Comparison': f'mRMR-MS vs {component}',
                    'Wins': wins,
                    'Ties': ties,
                    'Losses': losses,
                    'Win/tie rate': (wins + ties) / total if total else float('nan'),
                    'Mean delta': chunk['delta'].mean(),
                    'Median delta': chunk['delta'].median(),
                }
            )

    out = pd.DataFrame(rows)
    comparison_order = {'mRMR-MS vs MS': 0, 'mRMR-MS vs mRMR': 1}
    out['_dataset_order'] = _dataset_order_rank(out['dataset_name'])
    out['_comparison_order'] = out['Comparison'].map(comparison_order).fillna(len(comparison_order)).astype(int)
    out = out.sort_values(['_dataset_order', '_comparison_order']).drop(
        columns=['_dataset_order', '_comparison_order']
    )
    out['dataset_name'] = out['dataset_name'].map(DATASET_DISPLAY_NAMES).fillna(out['dataset_name'])
    return out.rename(columns={'dataset_name': 'Dataset'}).reset_index(drop=True)


def build_dataset_sample_size_recall_table(summary_df):
    method_order = ['MS', 'mRMR', 'mRMR_MS_linear']
    method_labels = {
        'MS': 'MS',
        'mRMR': 'mRMR',
        'mRMR_MS_linear': 'mRMR-MS',
    }
    working = summary_df.loc[summary_df['method_output_name'].isin(method_order)].copy()
    recalls = (
        working.groupby(['dataset_name', 'n_samples', 'method_output_name'], as_index=False)[
            'mean_recall_mb_grouped'
        ]
        .mean()
        .rename(columns={'mean_recall_mb_grouped': 'recall'})
    )
    recalls['method_display'] = recalls['method_output_name'].map(method_labels)
    table = recalls.pivot_table(
        index='dataset_name',
        columns=['n_samples', 'method_display'],
        values='recall',
    )
    table = table.reindex(index=DATASET_DISPLAY_ORDER)
    table = table.reindex(columns=pd.MultiIndex.from_product([[250, 1000, 5000], ['MS', 'mRMR', 'mRMR-MS']]))
    table = table.reset_index()
    table['dataset_name'] = table['dataset_name'].map(DATASET_DISPLAY_NAMES).fillna(table['dataset_name'])
    table.columns = pd.MultiIndex.from_tuples(
        [('Dataset', '')] + [(f'n={sample_size}', method) for sample_size, method in table.columns[1:]]
    )
    return table


profile_columns = [
    'dataset_name',
    'best_or_tie_rate',
    'strict_best',
    'tied_best',
    'not_best',
    'best_or_tie_250',
    'best_or_tie_1000',
    'best_or_tie_5000',
    'n_nodes',
    'mb_mean',
    'spouses_mean',
    'features_after_ohe_mean',
    'onehot_dummy_cols_mean',
]

mrmr_ms_profile_supp = _publishable_table(mrmr_dataset_profile_table.loc[:, profile_columns])

mrmr_ms_extremes_supp = _publishable_table(
    mrmr_dataset_profile_table.loc[:, profile_columns].assign(
        **{
            'Profile group': lambda df: df['dataset_name'].map(
                {
                    'link': 'Favorable',
                    'pathfinder': 'Favorable',
                    'ecoli70': 'Favorable',
                    'mehra-complete': 'Favorable',
                    'magic-niab': 'Less favorable',
                    'magic-irri': 'Less favorable',
                    'healthcare': 'Less favorable',
                    'arth150': 'Intermediate',
                    'sangiovese': 'Dense local',
                }
            ),
            'Method': 'mRMR-MS',
        }
    )[
        [
            'Profile group',
            'Method',
            'dataset_name',
            'best_or_tie_rate',
            'best_or_tie_250',
            'best_or_tie_1000',
            'best_or_tie_5000',
            'n_nodes',
            'mb_mean',
            'spouses_mean',
            'features_after_ohe_mean',
            'onehot_dummy_cols_mean',
        ]
    ]
)

mrmr_ms_component_pairwise_supp = build_component_pairwise_summary(summary_df)

mrmr_ms_recall_by_sample_size_supp = build_dataset_sample_size_recall_table(summary_df)

mrmr_ms_discussion_evidence_supp = (
    mrmr_dataset_profile_table.loc[
        mrmr_dataset_profile_table['dataset_name'].isin(MRMR_DISCUSSION_DATASETS),
        [
            'dataset_name',
            'best_or_tie_rate',
            'best_or_tie_250',
            'best_or_tie_1000',
            'best_or_tie_5000',
            'n_nodes',
            'mb_mean',
            'spouses_mean',
            'features_after_ohe_mean',
            'onehot_dummy_cols_mean',
        ],
    ]
    .assign(
        discussion_group=lambda df: df['dataset_name'].map(
            {
                'link': 'Favorable',
                'pathfinder': 'Favorable',
                'ecoli70': 'Favorable',
                'mehra-complete': 'Favorable',
                'magic-niab': 'Less favorable',
                'magic-irri': 'Less favorable',
                'arth150': 'Intermediate',
                'healthcare': 'Less favorable',
                'sangiovese': 'Dense local',
            }
        )
    )
)

component_rates_wide = (
    mrmr_ms_component_pairwise_supp
    .rename(columns={'Dataset': 'dataset_display'})
    .assign(dataset_name=lambda df: df['dataset_display'].map({v: k for k, v in DATASET_DISPLAY_NAMES.items()}))
    .pivot(index='dataset_name', columns='Comparison', values='Win/tie rate')
    .rename(columns={'mRMR-MS vs MS': 'win_or_tie_vs_MS', 'mRMR-MS vs mRMR': 'win_or_tie_vs_mRMR'})
    .reset_index()
)

mrmr_ms_discussion_evidence_supp = _publishable_table(
    mrmr_ms_discussion_evidence_supp.merge(component_rates_wide, on='dataset_name', how='left')
)

_print_latex_table(
    'supp_mrmr_ms_profile_all_datasets',
    mrmr_ms_profile_supp,
    'Dataset-level profile of mRMR-MS against its components MS and mRMR. Best/tie is the fraction of target--sample-size configurations in which mRMR-MS is either uniquely best or tied for the best grouped Markov-blanket recall within the triplet.',
    'tab:supp_mrmr_ms_profile_all_datasets',
    column_format='lrrrrrrrrrrrr',
    font_size='scriptsize',
)

_print_latex_table(
    'supp_mrmr_ms_regime_profiles',
    mrmr_ms_extremes_supp,
    'Dataset-level profiles used to support the discussion of favorable, less favorable, intermediate, and dense-local regimes for mRMR-MS against MS and mRMR.',
    'tab:supp_mrmr_ms_regime_profiles',
    column_format='lllrrrrrrrrr',
    font_size='scriptsize',
)

_print_latex_table(
    'supp_mrmr_ms_component_pairwise',
    mrmr_ms_component_pairwise_supp,
    'Pairwise dataset-level comparison of mRMR-MS against each component method. Wins, ties, and losses are counted at the target--sample-size configuration level using grouped Markov-blanket recall.',
    'tab:supp_mrmr_ms_component_pairwise',
    column_format='llrrrrrr',
    font_size='small',
)

_print_latex_table(
    'supp_mrmr_ms_recall_by_sample_size',
    mrmr_ms_recall_by_sample_size_supp,
    'Mean grouped Markov-blanket recall by dataset, sample size, and method for the focused comparison among MS, mRMR, and mRMR-MS.',
    'tab:supp_mrmr_ms_recall_by_sample_size',
    column_format='lrrrrrrrrr',
    font_size='scriptsize',
)

_print_latex_table(
    'supp_mrmr_ms_discussion_evidence',
    mrmr_ms_discussion_evidence_supp,
    'Focused evidence table for the datasets discussed in the text, combining mRMR-MS best/tie rates, pairwise win/tie rates against the two components, and structural descriptors of each Bayesian network.',
    'tab:supp_mrmr_ms_discussion_evidence',
    column_format='lrrrrrrrrrlrr',
    font_size='scriptsize',
)


===== supp_mrmr_ms_profile_all_datasets =====
\begin{table}[!htbp]
\centering
\scriptsize
\setlength{\tabcolsep}{4pt}
\renewcommand{\arraystretch}{1.08}
\caption{Dataset-level profile of mRMR-MS against its components MS and mRMR. Best/tie is the fraction of target--sample-size configurations in which mRMR-MS is either uniquely best or tied for the best grouped Markov-blanket recall within the triplet.}
\label{tab:supp_mrmr_ms_profile_all_datasets}
\begin{tabular}{lrrrrrrrrrrrr}
\toprule
Dataset & Best/tie & Strict best & Tied best & Not best & n=250 & n=1000 & n=5000 & Nodes & MB mean & Spouses mean & OHE features & OHE dummies \\
\midrule
Link & 0.706 & 3 & 9 & 5 & 0.333 & 1.000 & 0.800 & 724 & 12.000 & 4.800 & 609.200 & 1.200 \\
Pathfinder & 0.750 & 2 & 7 & 3 & 0.750 & 1.000 & 0.500 & 109 & 11.000 & 4.250 & 284.750 & 254.250 \\
Ecoli70 & 0.700 & 14 & 7 & 9 & 0.600 & 0.700 & 0.800 & 46 & 6.700 & 3.100 & 45.000 & 0.000 \\
Magic-NIAB & 0.467 & 12 & 2 & 16 & 0.500 & 0.500 & 0.400 & 44 &